# Churn Modelling with Artificial Neural Networks

This project demonstrates how to build and train an Artificial Neural Network (ANN) to predict customer churn using the Churn Modelling dataset.

**Dataset:**

The dataset contains information about bank customers and whether they have churned (exited the bank). The features include:

*   `RowNumber`, `CustomerId`, `Surname`: Identifiers (not used for training).
*   `CreditScore`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`, `EstimatedSalary`: Numerical features.
*   `Geography`, `Gender`: Categorical features.
*   `Exited`: Target variable (1 if the customer churned, 0 otherwise).

**Libraries Used:**

*   **pandas:** For data loading and manipulation.
*   **numpy:** For numerical operations.
*   **matplotlib.pyplot:** For plotting the training history.
*   **tensorflow:** For building and training the ANN model.
*   **sklearn:** For data splitting, feature scaling, and evaluation metrics.

**Methodology:**

1.  **Data Loading and Exploration:** The dataset is loaded using pandas, and the first few rows are displayed to understand the data structure.
2.  **Feature Engineering:** Categorical features (`Geography` and `Gender`) are one-hot encoded using `pd.get_dummies`. The original categorical columns are dropped from the feature set.
3.  **Data Splitting:** The dataset is split into training and testing sets using `train_test_split` from `sklearn.model_selection`.
4.  **Feature Scaling:** Numerical features are scaled using `StandardScaler` from `sklearn.preprocessing` to ensure that all features have a similar range, which helps the ANN converge faster.
5.  **ANN Model Building:**
    *   A `Sequential` model is initialized from `tensorflow.keras.models`.
    *   `Dense` layers are added to the model. The first layer is the input layer, followed by hidden layers with `relu` activation. `Dropout` layers are included after the input layer to prevent overfitting.
    *   The output layer has a single unit with `sigmoid` activation, suitable for binary classification.
6.  **Model Compilation:** The model is compiled with the `Adam` optimizer, `binary_crossentropy` loss function (for binary classification), and `accuracy` as the evaluation metric.
7.  **Early Stopping:** `EarlyStopping` callback from `tensorflow.keras.callbacks` is used to monitor the validation loss and stop training if it stops improving, preventing overfitting.
8.  **Model Training:** The model is trained on the training data using the `fit` method. A validation split is used to monitor performance on unseen data during training.
9.  **Model Evaluation:**
    *   Predictions are made on the test set using the trained model.
    *   A confusion matrix is generated using `confusion_matrix` from `sklearn.metrics` to evaluate the model's performance.
    *   The accuracy score is calculated using `accuracy_score` from `sklearn.metrics`.
    *   Training history (accuracy and loss) is plotted to visualize the model's learning process.

This project provides a comprehensive example of applying ANNs to a real-world classification problem, including data preprocessing, model building, training, and evaluation.

In [ ]:
pip install tensorflow

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
#import basic libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
dataset = pd.read_csv('Churn_Modelling.csv')
dataset.head()

In [ ]:
## Divide the dataset into denpended and indipendent features
X = dataset.iloc[:, 3:13]
y = dataset.iloc[:, 13]

In [ ]:
## ferature Engineering
geography = pd.get_dummies(X['Geography'], drop_first=True) #drop_first will give 2 column which give the all 3 data loke 00 01 10
gender = pd.get_dummies(X['Gender'], drop_first=True)

In [ ]:
X.drop(['Geography', 'Gender'], axis=1, inplace=True)

In [ ]:
X

In [ ]:
pd.concat([X, geography, gender], axis=1)

In [ ]:
## train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [ ]:
##feature scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
  X_train.shape

In [ ]:
##Part 2 Now create the ANN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU, PReLU, ELU, ReLU
from tensorflow.keras.layers import Dropout

In [ ]:
## lets initialize the ANN
classifier = Sequential()

## adding input layer
classifier.add(Dense(units=8, activation='relu'))

#adding dropout
classifier.add(Dropout(0.3))

In [ ]:
# addding the first hidden layer
classifier.add(Dense(units=7, activation='relu'))

# addding the second hidden layer
classifier.add(Dense(units=6, activation='relu'))

In [ ]:
#adding the output layer
classifier.add(Dense(units=1, activation='sigmoid'))

In [ ]:
classifier.compile(optimizer=opt,  loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
import tensorflow as tf
opt = tf.keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.95, epsilon=1e-07, amsgrad=False, name='Adam')

In [ ]:
#Early Stopping
import tensorflow as tf
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0.0001,
    patience=20,
    verbose=1,
    mode="auto",
    baseline=None,
    restore_best_weights=False,
    start_from_epoch=0,
)

In [ ]:
model_history = classifier.fit(X_train, y_train, validation_split=0.33, batch_size=10, epochs=1000, callbacks=early_stopping)

In [ ]:
model_history.history.keys()

In [ ]:
#summarize history for accuracy
plt.plot(model_history.history['accuracy'])
plt.plot(model_history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
#summarize the loss
plt.plot(model_history.history['loss'])
plt.plot(model_history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
#part 3 - making the prediction and evauating the model

y_pred = classifier.predict(X_test)
y_pred = (y_pred  >= 0.5)

In [ ]:
# make the confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

In [ ]:
cm

In [ ]:
#Calculate accuracy
from sklearn.metrics import accuracy_score
score = accuracy_score(y_pred, y_test)
score